# **VectorDB, Hugging Face & Ollama - Assignment**

**Question 1: What is a Vector Database (VectorDB) and how is it different from traditional databases?**

A Vector Database (VectorDB) is a specialized database designed to store, index, and retrieve high-dimensional vector embeddings—numerical representations of data (text, images, audio) that capture semantic meaning.

Unlike traditional databases that query exact matches or structured data, VectorDBs enable fast semantic similarity search (ANN), finding similar content based on mathematical distance.

**Question 2: Explain the various types of VectorDBs available and describe their suitability for different use cases.**

Vector databases, essential for semantic search and AI, fall into three main types based on deployment and functionality: specialized (Pinecone, Milvus, Weaviate, Qdrant), plugin-based (Pgvector for PostgreSQL), and hybrid solutions.

- Specialized databases (e.g., Pinecone, Milvus) excel in high-speed, large-scale search for RAG and AI applications, offering robust scalability for billions of vectors.

- Plugin-based databases (e.g., Pgvector) are ideal for developers adding semantic search to existing SQL infrastructures.

- Hybrid solutions (e.g., Elasticsearch, Redis) serve real-time analytics by merging vector search with traditional text/keyword search.

**Question 3: Why is Chroma DB considered important in the context of AI/ML projects? Describe its key features.**

Chroma DB is important in AI/ML projects because it is an open-source vector database specifically designed to store, manage, and search high-dimensional vector embeddings, which are crucial for AI applications to understand context and meaning from complex data.

It allows for efficient semantic search rather than just keyword matching, enabling more intuitive and powerful AI experiences like intelligent chatbots and accurate recommendation engines.

**Question 4: What are the benefits of using Hugging Face Hub for generative AI tasks?**

Hugging Face Hub offers a centralized, open-source repository of over 2 million pre-trained models and 500k+ datasets for generative AI. It enables rapid development by providing easy access to state-of-the-art models for text, image, and audio tasks, reducing the need for costly training from scratch.

**Question 5: Describe the process and advantages of navigating and using pre-trained models from the Hugging Face Hub.**

Navigating and using pre-trained models from the Hugging Face Hub involves a straightforward process and offers significant advantages.

Users can visit the Hugging Face website to browse a vast repository of models for various tasks, using filters to find the right one (e.g., for NLP, computer vision, audio processing). Each model has a "model card" that provides essential information, including its intended use, limitations, and code snippets. To use a model, users typically install the relevant libraries (like the Hugging Face Transformers library) and then, using a few lines of Python code, can load and utilize the model directly or fine-tune it with their own data.

The primary advantages of this approach are:

- Efficiency: Pre-trained models are already trained on large datasets, saving significant time and computational resources compared to training from scratch.

- Accessibility: The platform lowers the barrier to entry for complex AI development by providing easy access to state-of-the-art models and integration with frameworks like PyTorch and TensorFlow.

**Question 6: Install and set up Chroma DB, and insert sample vector data for semantic search.**

In [77]:
!pip install chromadb

  Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl.metadata (2.5 kB)
Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl (231 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry

In [78]:
import chromadb
client = chromadb.Client()

In [79]:
import chromadb
# Data will be stored in a directory named "chroma_data"
client = chromadb.PersistentClient(path="./chroma_data")

In [80]:
collection.add(
    documents=[
        "This is a document about machine learning.",
        "A second document about artificial intelligence.",
        "The third document is about data science.",
        "This last document is about natural language processing."
    ],
    metadatas=[
        {"topic": "machine learning"},
        {"topic": "AI"},
        {"topic": "data science"},
        {"topic": "NLP"}
    ],
    ids=[
        "doc1",
        "doc2",
        "doc3",
        "doc4"
    ]
)

In [81]:
results = collection.query(
    query_texts=["What is data science?"],
    n_results=2 # Return the 2 most similar results
)

print(results)

{'ids': [['doc3', 'doc1']], 'embeddings': None, 'documents': [['The third document is about data science.', 'This is a document about machine learning.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'topic': 'data science'}, {'topic': 'machine learning'}]], 'distances': [[0.8497763872146606, 1.15921950340271]]}


**Question 7: Demonstrate how to download and fine-tune a Hugging Face model for a text generation task.**

In [75]:
!pip install transformers datasets accelerate torch

In [76]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [82]:
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling

# Create a dummy data.txt file for demonstration if it doesn't exist
with open("data.txt", "w") as f:
    f.write("This is a sample text for fine-tuning. It contains some example sentences. Generative models learn from such data. We are fine-tuning a GPT-2 model.")
    f.write("\nMore text for the training data. This will help the model generate relevant sentences. Hopefully, it learns well.")

# Load dataset (example: text file)
dataset = load_dataset("text", data_files={"train": "data.txt"})

# Set the padding token for the tokenizer
tokenizer.pad_token = tokenizer.eos_token

# Tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [83]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

In [84]:
trainer.train()
trainer.save_model("./gpt2-finetuned")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [85]:
from transformers import pipeline

generator = pipeline('text-generation', model='./gpt2-finetuned', tokenizer=model_name)
print(generator("Once upon a time", max_length=50))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Once upon a time, I was in a position where I had to decide whether I was going to be a feminist or not. I was going to make sure that I was taking a position in which I was not interested in getting attention from anyone. It was my job to make sure that I was not a feminist. I also had to remember to not be a misogynist.\n\nI was very lucky that I had the right person who was smart and smart enough to realize that I was not a feminist. But it was a big mistake. We all made mistakes before. You don't get to decide if you are a feminist or not, but you have to be a feminist. Your job is not to have a sense of what to do.\n\nFor me, I'm still very much a feminist. I'm not a racist. I'm a proud feminist. I'm a happy feminist. I've been proud of my work for years and I'm still proud of it. I think that all of us should be proud of it. But I think that it is important that we not be afraid of being a racist or sexist. We should be proud of what we do and how we do it. I

**Question 8: Create a custom LLM using Ollama and Llama2, and run it locally for basic text prompts.**

In [86]:
modelfile_content = """FROM llama2\n\nPARAMETER temperature 0.7\n\nSYSTEM \"\"\"You are a helpful and concise AI assistant.\nAlways provide clear answers to text prompts.\"\"\"\n"""

# Save the Modelfile content to a file
with open("Modelfile_custom", "w") as f:
    f.write(modelfile_content)

# Create the custom LLM using Ollama
# Ensure Ollama is installed and running
!ollama create custom-llama2 -f Modelfile_custom

Error: could not connect to ollama server, run 'ollama serve' to start it


In [87]:
!ollama create my-custom-llama2 -f Modelfile

Error: could not connect to ollama server, run 'ollama serve' to start it


In [88]:
!ollama run my-custom-llama2

Error: could not connect to ollama server, run 'ollama serve' to start it


**Question 9: Implement a basic RAG (Retrieval-Augmented Generation) system using Ollama with Llama3.**

In [89]:
!ollama pull llama3

Error: could not connect to ollama server, run 'ollama serve' to start it


In [90]:
!pip install langchain langchain-community langchain-chroma

  Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl.metadata (2.5 kB)
Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl (231 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry

In [91]:
!pip install --upgrade langchain langchain-community langchain-chroma langchain-core langchain-text-splitters bs4

  Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl.metadata (2.5 kB)
Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl (231 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry

In [92]:
!ollama pull nomic-embed-text

Error: could not connect to ollama server, run 'ollama serve' to start it


In [93]:
!pip install langchain langchain-community ollama chromadb

  Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl.metadata (2.5 kB)
Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl (231 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry

In [94]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a dummy data.txt file for demonstration if it doesn't exist
with open("your_data.txt", "w") as f:
    f.write("This is a sample medical research article. It discusses recent advancements in AI for diagnosing diseases early. The article highlights the importance of large datasets and advanced machine learning models.")
    f.write("\nAnother article talks about new drug discoveries and their impact on chronic illnesses. Clinical trials are underway to validate their efficacy and safety.")

# Load your local data
loader = TextLoader("your_data.txt")
data = loader.load()

# Split into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(data)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content[:100]}...")

Number of chunks: 1
Chunk 1: This is a sample medical research article. It discusses recent advancements in AI for diagnosing dis...


**Question 10: A health-tech startup wants to build a chatbot that can answer user queries based on medical research articles. Propose and explain a solution using Hugging Face models for understanding, VectorDB for retrieval, and Ollama for generation.**

The startup can build a Retrieval-Augmented Generation (RAG) pipeline to ensure accuracy and reduce hallucinations.

First, use a Hugging Face embedding model (like sentence-transformers/all-MiniLM-L6-v2) to convert medical research articles into numerical vectors. These vectors are stored in a VectorDB (such as Pinecone or Milvus) for efficient similarity searching.

When a user asks a question, the system retrieves the most relevant snippets from the VectorDB and passes them as context to a local LLM (like Llama 3 or Mistral) running on Ollama. Ollama then generates a natural language response strictly grounded in the retrieved medical data.